In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Data/WELFake_News_Data.csv')

In [3]:
df

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
2,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
3,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
4,5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,1
...,...,...,...,...
71532,72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0
71533,72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1
71534,72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0
71535,72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0


In [4]:
from sklearn.model_selection import train_test_split
train_set, temp_set = train_test_split(df, test_size=0.2, random_state=42)

In [5]:
val_set, test_set = train_test_split(temp_set, test_size=0.5, random_state=42)

In [7]:
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_set.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_set.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_set.reset_index(drop=True))

In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenizer_fn(example):
  return tokenizer(example['text'], truncation=True, padding="max_length", max_length=512)
train_dataset = train_dataset.map(tokenizer_fn, batched=True)
val_dataset = val_dataset.map(tokenizer_fn, batched=True)
test_dataset = test_dataset.map(tokenizer_fn, batched=True)


Map:   0%|          | 0/57229 [00:00<?, ? examples/s]

Map:   0%|          | 0/7154 [00:00<?, ? examples/s]

Map:   0%|          | 0/7154 [00:00<?, ? examples/s]

In [10]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [12]:
def compute_metrics(eval_pred):
  logits, lables = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return {
      "accuracy": accuracy_score(lables, predictions),
      "f1": f1_score(lables, predictions)
  }

In [14]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.037237,0.050875,0.982248,0.982145
2,0.000143,0.034443,0.994129,0.994172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14308, training_loss=0.04508108343446858, metrics={'train_runtime': 1842.4892, 'train_samples_per_second': 62.121, 'train_steps_per_second': 7.766, 'total_flos': 1.5161953515368448e+16, 'train_loss': 0.04508108343446858, 'epoch': 2.0})